
# Analyse de sentiments avec un modèle Transformers

Ce notebook reprend ton script, mais avec **plus d'explications** et une structure plus pédagogique.

## Objectif
On veut entraîner un modèle de type BERT / RoBERTa pour faire de la **classification binaire de sentiments** :

- **0** : avis négatif
- **1** : avis positif

Le pipeline complet sera :

1. charger les données ;
2. vérifier leur structure ;
3. tokenizer les textes ;
4. créer les jeux d'entraînement et de test ;
5. charger un modèle pré-entraîné ;
6. fine-tuner le modèle ;
7. mesurer les performances.

## Remarque importante
Dans ton script original, il y avait un point délicat :

- `model_name = "haisongzhang/roberta-tiny-cased"`
- puis `BertForSequenceClassification.from_pretrained(model_name)`

Comme le modèle choisi est de type **RoBERTa** et non **BERT**, il est plus sûr d'utiliser :

- `AutoTokenizer`
- `AutoModelForSequenceClassification`

Cela permet de charger automatiquement la bonne architecture selon le modèle choisi.


In [1]:

# Si besoin, décommente cette ligne dans un notebook vide
# %pip install -U transformers datasets scikit-learn tqdm


In [2]:

import json
import gc
import numpy as np
import torch
from collections import Counter
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import torch.optim as optim



## 1. Vérification du matériel

On regarde si un **GPU CUDA** est disponible.  
Si oui, l'entraînement sera beaucoup plus rapide.


In [3]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))


Device utilisé : cuda
Nom du GPU : NVIDIA GeForce RTX 2070



## 2. Téléchargement et chargement des données

Le fichier `json_pol.json` contient une liste d'exemples sous la forme :

```python
[
    ["texte de l'avis", label],
    ...
]
```

où `label` vaut :

- `0` pour négatif
- `1` pour positif


In [4]:

# Si le fichier n'est pas encore présent, on peut le télécharger
# !wget https://thome.isir.upmc.fr/classes/RITAL/json_pol.json


In [ ]:
from load_data import load_movies

data_path = "../datasets/"
path = data_path + "movies1000/"
alltxts, y = load_movies(path)

In [6]:
# file = "./json_pol.json"

# with open(file, encoding="utf-8") as f:
#     data = json.load(f)

# counter = Counter(x[1] for x in data)

# print("Nombre total d'avis :", len(data))
# print("Nombre d'avis positifs :", counter[1])
# print("Nombre d'avis négatifs :", counter[0])
# print("\nPremier exemple :")
# print(data[0])



## 3. Choix du modèle et du tokenizer

On choisit ici un petit modèle pour aller plus vite :

```python
haisongzhang/roberta-tiny-cased
```

C'est une bonne option pour tester le pipeline rapidement.

Le **tokenizer** transforme un texte en entiers exploitables par le modèle.

### Deux sorties importantes du tokenizer

- `input_ids` : les indices des tokens dans le vocabulaire ;
- `attention_mask` : indique quels tokens doivent être pris en compte :
  - `1` : token réel
  - `0` : padding


In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-cased"
# model_name = "bert-base-cased"
# model_name = "haisongzhang/roberta-tiny-cased"

# Chargement automatique
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print("Modèle utilisé :", model_name)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modèle utilisé : distilbert-base-cased



## 4. Test du tokenizer sur un exemple

On applique le tokenizer sur le premier texte du dataset.

### Paramètres utiles

- `max_length=512` : taille maximale de la séquence ;
- `truncation=True` : coupe les textes trop longs ;
- `padding="max_length"` : ajoute du padding pour que toutes les séquences aient la même longueur ;
- `return_tensors="pt"` : renvoie des tenseurs PyTorch.


In [8]:
maxL = 512

# sample_text = data[0][0]
sample_text = alltxts[0]
encoded_sample = tokenizer(
    sample_text,
    return_tensors="pt",
    max_length=maxL,
    truncation=True,
    padding="max_length",
)

print("Clés retournées :", encoded_sample.keys())
print("Shape input_ids :", encoded_sample["input_ids"].shape)
print("Shape attention_mask :", encoded_sample["attention_mask"].shape)

Clés retournées : KeysView({'input_ids': tensor([[  101, 17462, 22950,  1161,  1110,   170,  2633, 12020,  3205,   119,
          1103,  3670,  2603, 17233,  3687, 27262,  1104,  6968,   113,  2324,
          1476, 17462, 22950,  1389,  5860,  1106,  1141,  1821, 26237,  1389,
           114,   117,  1103,  2343,  1110,   170,  8431,  2221,   117,  1103,
          1586,  1110,  3258,  1133, 14756, 20641,   119,  1105,  1112,  8668,
          1110, 16679, 17462, 22950,  1161,   112,   188,  4583,  2597,   117,
         16714,  3873,  1105,  1817,  1114,   170,  2305,  1104, 27475,   119,
          1297,  1105,  6695,  2274,   170, 10488,  1183,  1440,  1120,  1297,
          1656,  1142, 24034,  5909,  9980,  1583,   119,  1229,  2999, 17221,
          1321,  1103, 21718,  6071,   188, 18062, 15796,  1468,  3136,  1104,
         10311,  4429,  1104, 20285,  1482,  1114,  7591,  2365,  1390,   117,
          1142,  1273, 15889,  1103, 15403,  1114,   170, 25322,  1490,  5909,
          1

In [9]:
print("Quelques input_ids :")
print(encoded_sample["input_ids"][0][:30])

print("\nQuelques valeurs du attention_mask :")
print(encoded_sample["attention_mask"][0][:30])

Quelques input_ids :
tensor([  101, 17462, 22950,  1161,  1110,   170,  2633, 12020,  3205,   119,
         1103,  3670,  2603, 17233,  3687, 27262,  1104,  6968,   113,  2324,
         1476, 17462, 22950,  1389,  5860,  1106,  1141,  1821, 26237,  1389])

Quelques valeurs du attention_mask :
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1])



## 5. Tokenization de tout le dataset

On encode maintenant **tous les textes** du corpus.

Cela peut prendre un peu de temps, car chaque phrase est transformée en tenseurs de taille fixe.


In [10]:

inputs_tokens = []
attention_masks = []

# for i in tqdm(range(len(data)), desc="Tokenisation"):
#     text = data[i][0]
for i in tqdm(range(len(alltxts)), desc="Tokenisation"):
    text = alltxts[i]
    encoded = tokenizer(
        text,
        return_tensors="pt",
        max_length=maxL,
        truncation=True,
        padding="max_length"
    )
    inputs_tokens.append(encoded["input_ids"])
    attention_masks.append(encoded["attention_mask"])

inputs_tokens = torch.cat(inputs_tokens, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)

print("Shape inputs_tokens :", inputs_tokens.shape)
print("Shape attention_masks :", attention_masks.shape)


Tokenisation:   0%|          | 0/2000 [00:00<?, ?it/s]

Shape inputs_tokens : torch.Size([2000, 512])
Shape attention_masks : torch.Size([2000, 512])



## 6. Création du vecteur des labels

Les labels doivent être associés à chaque texte.

Pour une classification binaire avec `CrossEntropyLoss`, on utilise des **entiers** :

- `0`
- `1`

On les stocke donc dans un tenseur de type `long`.


In [11]:

# y = torch.tensor([item[1] for item in data], dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print("Shape y :", y.shape)
print("Exemples de labels :", y[:10])


Shape y : torch.Size([2000])
Exemples de labels : tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])



## 7. Séparation entraînement / test

On coupe ici le dataset en deux parties :

- **50 % entraînement**
- **50 % test**

Dans un vrai projet, on pourrait aussi ajouter un jeu de **validation**.


In [12]:
rs = 10  # random_state pour la reproductibilité

(
    inputs_tokens_train,
    inputs_tokens_test,
    attention_masks_train,
    attention_masks_test,
    y_train,
    y_test,
) = train_test_split(
    inputs_tokens, attention_masks, y, test_size=0.7, random_state=rs, shuffle=True, stratify=y
)

print("Train input_ids :", inputs_tokens_train.shape)
print("Test input_ids  :", inputs_tokens_test.shape)
print("Train masks     :", attention_masks_train.shape)
print("Test masks      :", attention_masks_test.shape)
print("Train labels    :", y_train.shape)
print("Test labels     :", y_test.shape)

Train input_ids : torch.Size([1000, 512])
Test input_ids  : torch.Size([1000, 512])
Train masks     : torch.Size([1000, 512])
Test masks      : torch.Size([1000, 512])
Train labels    : torch.Size([1000])
Test labels     : torch.Size([1000])



## 8. Création des objets `TensorDataset` et `DataLoader`

`TensorDataset` regroupe les entrées et les labels.

Chaque élément du dataset contient :

1. `input_ids`
2. `attention_mask`
3. `label`

Puis `DataLoader` permet de lire les données **par mini-batchs**, ce qui est indispensable pour l'entraînement.


In [13]:

dataset_train = TensorDataset(inputs_tokens_train, attention_masks_train, y_train)
dataset_test = TensorDataset(inputs_tokens_test, attention_masks_test, y_test)

batch_size = 16

train_dataloader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)

print("Nombre de batchs train :", len(train_dataloader))
print("Nombre de batchs test  :", len(test_dataloader))


Nombre de batchs train : 63
Nombre de batchs test  : 63



## 9. Chargement du modèle de classification

On charge ici directement un modèle pré-entraîné pour la **classification de séquence**.

Comme on a deux classes, on met :

```python
num_labels=2
```

Ensuite, le modèle produira pour chaque texte un vecteur de taille 2 contenant les **logits** des deux classes.


In [14]:

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.to(device)
print(model.__class__.__name__)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification



## 10. Définition d'une fonction d'évaluation

Cette fonction calcule l'accuracy sur un `DataLoader`.

### Principe
Pour chaque batch :

1. on fait une passe avant ;
2. on récupère les logits ;
3. on prend la classe de score maximal avec `argmax` ;
4. on compare à la vérité terrain.


In [15]:
def evaluate_accuracy(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            outputs = model(input_ids=b_input_ids, attention_mask=b_input_mask)
            preds = outputs.logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())

    if len(all_labels) > 0:
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='binary')
        prec = precision_score(all_labels, all_preds, average='binary')
        rec = recall_score(all_labels, all_preds, average='binary')
    else:
        acc = 0.0
        f1 = 0.0
        prec = 0.0
        rec = 0.0
    
    return {
        'accuracy': acc,
        'f1_score': f1,
        'precision': prec,
        'recall': rec
    }



## 11. Préparation de l'entraînement

On utilise ici :

- **AdamW** via `torch.optim.AdamW` pour l'optimisation ;
- un **scheduler linéaire** pour ajuster progressivement le taux d'apprentissage ;
- le calcul de la loss directement via `labels=...` dans l'appel du modèle.

### Pourquoi `labels=...` ?
Les modèles `AutoModelForSequenceClassification` savent calculer eux-mêmes la bonne loss de classification, ce qui simplifie le code.


In [16]:

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

epochs = 10
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print("Nombre total de steps :", total_steps)


Nombre total de steps : 630



## 12. Boucle d'entraînement

À chaque batch :

1. on envoie les données sur le bon device ;
2. on remet les gradients à zéro ;
3. on calcule la loss ;
4. on fait la rétropropagation ;
5. on met à jour les poids ;
6. on met à jour le scheduler.

On affiche ensuite :

- la loss moyenne d'entraînement ;
- l'accuracy sur train ;
- l'accuracy sur test.


In [19]:
avg_train_loss = 1
for epoch in range(epochs):
    if avg_train_loss > 0.05:
        model.train()
        total_loss = 0.0
    
        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}")
    
        for batch in progress_bar:
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)
    
            optimizer.zero_grad()
    
            outputs = model(
                input_ids=b_input_ids, attention_mask=b_input_mask, labels=b_labels
            )
    
            loss = outputs.loss
            total_loss += loss.item()
    
            loss.backward()
            optimizer.step()
            scheduler.step()
    
            progress_bar.set_postfix(loss=loss.item())
    
        avg_train_loss = total_loss / len(train_dataloader)
        train_metrics = evaluate_accuracy(model, train_dataloader, device)
        test_metrics = evaluate_accuracy(model, test_dataloader, device)
    
        print(f"\nEpoch {epoch+1}")
        print(f"Loss moyenne train : {avg_train_loss:.4f}")
        print("Train")
        print(f"Accuracy     : {train_metrics['accuracy']:.4f}")
        print(f"F1           : {train_metrics['f1_score']:.4f}")
        print(f"Precision    : {train_metrics['precision']:.4f}")
        print(f"Recall       : {train_metrics['recall']:.4f}")
        print("Test")
        print(f"Accuracy     : {test_metrics['accuracy']:.4f}")
        print(f"F1           : {test_metrics['f1_score']:.4f}")
        print(f"Precision    : {test_metrics['precision']:.4f}")
        print(f"Recall       : {test_metrics['recall']:.4f}")

Epoch 1/10:   0%|          | 0/63 [00:00<?, ?it/s]


Epoch 1
Loss moyenne train : 0.2815
Train
Accuracy     : 0.9580
F1           : 0.9568
Precision    : 0.9852
Recall       : 0.9300
Test
Accuracy     : 0.8060
F1           : 0.7958
Precision    : 0.8400
Recall       : 0.7560


Epoch 2/10:   0%|          | 0/63 [00:00<?, ?it/s]


Epoch 2
Loss moyenne train : 0.1350
Train
Accuracy     : 0.9950
F1           : 0.9950
Precision    : 0.9980
Recall       : 0.9920
Test
Accuracy     : 0.8100
F1           : 0.8017
Precision    : 0.8384
Recall       : 0.7680


Epoch 3/10:   0%|          | 0/63 [00:00<?, ?it/s]


Epoch 3
Loss moyenne train : 0.0552
Train
Accuracy     : 0.9990
F1           : 0.9990
Precision    : 1.0000
Recall       : 0.9980
Test
Accuracy     : 0.8110
F1           : 0.8258
Precision    : 0.7658
Recall       : 0.8960


Epoch 4/10:   0%|          | 0/63 [00:00<?, ?it/s]


Epoch 4
Loss moyenne train : 0.0240
Train
Accuracy     : 0.9990
F1           : 0.9990
Precision    : 1.0000
Recall       : 0.9980
Test
Accuracy     : 0.8120
F1           : 0.8124
Precision    : 0.8108
Recall       : 0.8140



## 13. Prédiction sur quelques exemples

On peut maintenant tester le modèle sur quelques phrases pour voir ce qu'il prédit.

### Interprétation
- classe `0` : négatif
- classe `1` : positif


In [20]:

def predict_sentiment(text, model, tokenizer, device, max_length=512):
    model.eval()
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = probs.argmax(dim=1).item()

    return pred, probs.squeeze().cpu().numpy()


In [21]:

examples = [
    "I absolutely loved this movie, it was amazing.",
    "This film was boring and too long.",
    "Not bad, but not great either."
]

for text in examples:
    pred, probs = predict_sentiment(text, model, tokenizer, device, maxL)
    print("Texte :", text)
    print("Classe prédite :", pred, "(0=négatif, 1=positif)")
    print("Probabilités :", probs)
    print("-" * 80)


Texte : I absolutely loved this movie, it was amazing.
Classe prédite : 0 (0=négatif, 1=positif)
Probabilités : [0.9474233  0.05257669]
--------------------------------------------------------------------------------
Texte : This film was boring and too long.
Classe prédite : 1 (0=négatif, 1=positif)
Probabilités : [0.00982839 0.9901716 ]
--------------------------------------------------------------------------------
Texte : Not bad, but not great either.
Classe prédite : 1 (0=négatif, 1=positif)
Probabilités : [0.25248718 0.7475128 ]
--------------------------------------------------------------------------------



## 14. Sauvegarde du modèle fine-tuné

Une fois l'entraînement terminé, on peut enregistrer :

- le modèle ;
- le tokenizer.

Cela permet de les recharger plus tard sans refaire tout l'entraînement.


In [22]:
# save_dir = "./tokeniser_saved_model/" + model_name

# model.save_pretrained(save_dir)
# tokenizer.save_pretrained(save_dir)

# print("Modèle sauvegardé dans :", save_dir)


## 15. Conclusion

Dans ce notebook, on a vu comment :

- charger un dataset d'avis ;
- tokenizer les textes ;
- préparer les tenseurs PyTorch ;
- fine-tuner un modèle Transformers ;
- évaluer son accuracy ;
- faire des prédictions sur de nouveaux textes.

## Différences principales par rapport au script d'origine

1. **Plus d'explications** à chaque étape.
2. Utilisation de `AutoModelForSequenceClassification` pour éviter les incompatibilités entre architecture et nom du modèle.
3. Labels directement en `torch.long`.
4. Calcul de la loss via `labels=b_labels`.
5. Fonction d'évaluation plus générale, qui calcule l'accuracy sur n'importe quel `DataLoader`.

## Pistes d'amélioration

- ajouter un jeu de validation ;
- calculer précision / rappel / F1-score ;
- tester plusieurs modèles ;
- réduire `max_length` pour accélérer l'entraînement ;
- utiliser `Trainer` de Hugging Face pour un code plus compact.
